In [0]:
# import libraries

from pyspark.sql import SparkSession, functions as F

from pyspark.sql.functions import (
    explode, desc,  row_number, col, try_divide, year, try_to_date, count, 
    countDistinct
)

from pyspark.sql.window import Window

# import pandas as pd


In [0]:
# curl https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# record a file in local wget -O steam_game_output.json https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json

In [0]:
# from pyspark.sql import SparkSession

filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df = spark.read.format('json').load(filepath)

In [0]:
# Noumber of elements in dataframe
print(f"Entries number : {df.count()}")


Entries number : 55691


The dataset is to big for json_normalize method

In [0]:
type(df)

pyspark.sql.connect.dataframe.DataFrame

In [0]:
df.take(1)

[Row(data=Row(appid=10, categories=['Multi-player', 'Valve Anti-Cheat enabled', 'Online PvP', 'Shared/Split Screen PvP', 'PvP'], ccu=13990, developer='Valve', discount='0', genre='Action', header_image='https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513', initialprice='999', languages='English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean', name='Counter-Strike', negative=5199, owners='10,000,000 .. 20,000,000', platforms=Row(linux=True, mac=True, windows=True), positive=201215, price='999', publisher='Valve', release_date='2000/11/1', required_age='0', short_description="Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.", tags=Row(1980s=266, 1990's=1191, 2.5

In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

We observe 23 different values : data is a only a node, categories, platforms and tags contain nested informations. Let's flatened the dataframe to range datas at same level. 

In [0]:
# from pyspark.sql.functions import explode, desc

flat_df = df.select("id","data.*", explode("data.categories").alias("category"))
flat_df.display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,Multi-player
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's nu

In [0]:
# from pyspark.sql import functions as F

flat_df = flat_df.withColumn(
    "platform",
    F.concat_ws(', ', *[F.when(F.col(f"platforms.{f}"), F.lit(f)) for f in ['linux', 'mac', 'windows']])
)
flat_df.display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,Multi-player,"linux, mac, windows"
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,

In [0]:
flat_df = flat_df.withColumn("tag", F.to_json(F.col("tags")))
flat_df.display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform,tag
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,Multi-player,"linux, mac, windows","{""1980s"":266,""1990's"":1191,""Action"":5426,""Assassin"":227,""Classic"":2784,""Competitive"":1607,""FPS"":4831,""First-Person"":1707,""Military"":632,""Multiplayer"":3392,""Nostalgia"":131,""Old School"":769,""PvP"":881,""Score Attack"":289,""Shooter"":3353,""Strategy"":614,""Survival"":304,""Tactical"":1344,""Team-Based"":1864,""e-sports"":1192}"
10,10,"List(Multi-player

In [0]:
flat_df = flat_df.drop( "categories", "platforms", "tags")

In [0]:
flat_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string (nullable = true)
 |-- category: string (nullable = true)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)



In [0]:
# flat_df.write.csv("df.csv", header=True)

Export a csv with the .write method requires missing write right on working directory in databricks free edition. So, let's use pandas method .tocsv()

In [0]:
# import pandas as pd

# flat_df.toPandas().to_csv("df.csv", index=False)

# Explorary dataset analysis

## 1. Explore dataset

In [0]:
len(flat_df.columns)

23

In [0]:
flat_df.count()

191270

The dataframe has 23 columns and 191270 rows.

Now, let's query on the dataframe. Spark lets us run classic SQL queries on your tables, however, using classic SQL in Spark requires you to load the data in memory before running any query. We will use the .createOrReplaceTempView Spark DataFrame method in order to load the data in memory under a certain table name, we will then be able to run SQL queries on it.

In [0]:
flat_df.createOrReplaceTempView('table') # Creates a temporary view of the spark dataframe table in memory under the name
# my_table, which we can now query!

The .sql method lets you write queries in SQL while benefiting from the distributed computing advantages of Spark.

In [0]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-------

We observe some problems :
- unused spaces -> sort columns to verify wether it's a display problem or requests problem 
- different spelling : request problem on string
- empty entries : '' on website and platform -> verify other columns
- unused columns : id and appid are identifyers, we use name and can delete them, website, header_image 

We must verify with steam team before treat these data.
- noncompliant entries : are None, ., CD PROJEKT RED, [2.21] real publishers ?
- asiatic characters. We need information even if the langage changes.

Let's sample and observe values on selected columns.

In [0]:
flat_df.select('developer', 'publisher', 'owners', 'ccu', 'website', 'header_image').sample(fraction=0.0001).distinct().show(truncate=False)

+-------------------------------------------+---------------------------------+----------------------+-----+----------------------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|developer                                  |publisher                        |owners                |ccu  |website                                                                                 |header_image                                                                 |
+-------------------------------------------+---------------------------------+----------------------+-----+----------------------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|Ben Schwartz                               |Ben Schwartz                     |0 .. 20,000           |0    |                                                                

In [0]:
spark.sql("select * from table limit 1").show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|    category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,000,000 .. 20,...|  201215

In [0]:
result = spark.sql("select distinct name from table where name like'Counter-Strike%'")
result.show(truncate=False)

+--------------------------------+
|name                            |
+--------------------------------+
|Counter-Strike Nexon: Studio    |
|Counter-Strike: Global Offensive|
|Counter-Strike: Source          |
|Counter-Strike: Condition Zero  |
|Counter-Strike                  |
+--------------------------------+



In [0]:
flat_df.columns

['id',
 'appid',
 'ccu',
 'developer',
 'discount',
 'genre',
 'header_image',
 'initialprice',
 'languages',
 'name',
 'negative',
 'owners',
 'positive',
 'price',
 'publisher',
 'release_date',
 'required_age',
 'short_description',
 'type',
 'website',
 'category',
 'platform',
 'tag']

In [0]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = flat_df.filter(col(c) == '').count()
    if count > 0:
        print(f"{c}: {count}")

developer: 332
genre: 362
languages: 18
publisher: 433
release_date: 306
short_description: 52
website: 69054


There are 191270 rows. Website misses on one third rows. Other missing values should be converted.


## 2. Clean data

### a) treatments

Let's clean datas. First, delete unused spaces with trim method.

In [0]:
from pyspark.sql.functions import trim
clean_df = spark.sql("SELECT * FROM table").select([trim(col(c)).alias(c) for c in spark.sql("SELECT * FROM table").columns])
flat_df.createOrReplaceTempView('table')
spark.sql("SELECT * FROM table").show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-------

Then, lowercase all values.

In [0]:
from pyspark.sql.functions import lower, col

columns = [['id', 'appid', 'ccu', 'developer', 'discount', 'genre', 'header_image', 'initialprice', 'languages', 'name', 'negative', 'owners', 'positive', 'price', 'publisher', 'release_date', 'required_age', 'short_description', 'type', 'website', 'category', 'platform', 'tag']]

clean_df = clean_df \
    .select( \
    [lower(col(c)).alias(c) for c in columns[0]] \
    )
clean_df.show(5)

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    valve|       0|action|https://cdn.akama...|         999|english, french, ...|counter-strike|    5199|10,00

Next, drop unused columns.

In [0]:
clean_df = clean_df.drop('id', 'appid', 'header_image', 'website')

# TO DO Endly, deal with latest missing values.

### b) verify clean_df dataframe

In [0]:
clean_df.select('developer', 'publisher', 'owners', 'ccu').sample(fraction=0.0001).distinct().show(truncate=False)

+------------------------+------------------------------+--------------------+---+
|developer               |publisher                     |owners              |ccu|
+------------------------+------------------------------+--------------------+---+
|kinsei games            |kinsei games                  |50,000 .. 100,000   |0  |
|creativer game studio   |creativer game studio         |0 .. 20,000         |0  |
|evil turtle productions |evil turtle productions       |0 .. 20,000         |3  |
|eternal night studios   |eternal night studios         |0 .. 20,000         |0  |
|illfonic                |playstation pc llc            |100,000 .. 200,000  |217|
|large visible machine   |large visible machine         |20,000 .. 50,000    |0  |
|inters media            |inters media                  |0 .. 20,000         |0  |
|magic design studios    |magic design studios          |50,000 .. 100,000   |8  |
|virtual play            |virtual play                  |0 .. 20,000         |0  |
|hol

# TO DO Verify missing values

## 3. Analysis at the "macro" level

Which publisher has released the most games on Steam?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg(
        count('name').alias('game_nb'),
        countDistinct('release_date').alias('release_nb')
        ) \
    .orderBy(desc('release_nb')) \
    .limit(1)
result.show()

+--------------+-------+----------+
|     publisher|game_nb|release_nb|
+--------------+-------+----------+
|big fish games|    423|       405|
+--------------+-------+----------+



What is Valve's position as publisher ?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------+----------+----+
|     publisher|game_nb|release_nb|rank|
+--------------+-------+----------+----+
|big fish games|    423|       405|   1|
|         valve|     35|        30| 104|
+--------------+-------+----------+----+



How many games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['name']) \
    .distinct() \
    .count() 
print(f"Games number : {result}")

Games number : 54374


How many developers indicated on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer']) \
    .distinct() \
    .count()
print(f"Developer number : {result}")



Developer number : 34076


Which developer most contribute to games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer'],'name') \
    .distinct() \
    .groupBy('developer') \
    .agg(count('name').alias('game_nb')) \
    .orderBy(desc('game_nb')) \
    .limit(5)
result.show(truncate=False)

+------------------------+-------+
|developer               |game_nb|
+------------------------+-------+
|choice of games         |140    |
|creobit                 |120    |
|                        |117    |
|laush dmitriy sergeevich|108    |
|sokpop collective       |98     |
+------------------------+-------+



What are the best rated games?

In [0]:
result = clean_df \
    .select(clean_df['name'],'positive') \
    .groupBy('name') \
    .agg(F.sum('positive').alias('positive_nb')) \
    .orderBy(desc('positive_nb')) \
    .limit(10)
result.show()

+--------------------+-----------+
|                name|positive_nb|
+--------------------+-----------+
|counter-strike: g...|6.5376795E7|
|         garry's mod| 1.464108E7|
|  grand theft auto v|1.3521915E7|
|            terraria|1.3191243E7|
|       left 4 dead 2| 1.287672E7|
|     team fortress 2|1.1849698E7|
|              dota 2|1.0744265E7|
|tom clancy's rain...| 1.037201E7|
|                rust|1.0255189E7|
|       rocket league|  9929980.0|
+--------------------+-----------+



In [0]:
from pyspark.sql.functions import col, try_divide

result = flat_df \
    .select(flat_df['name'],'positive', 'negative') \
    .groupBy('name') \
    .agg( \
        F.sum('positive').alias('positive_sum'), \
        F.sum('negative').alias('negative_sum') \
        ) \
    .filter((col('positive_sum') > 0) & (col('negative_sum') > 0)) \
    .withColumn('ratio', col('positive_sum') /  col('negative_sum')) \
    .orderBy(desc('positive_sum')) \
    .limit(10)
result.show(truncate=False)

+--------------------------------+------------+------------+------------------+
|name                            |positive_sum|negative_sum|ratio             |
+--------------------------------+------------+------------+------------------+
|Counter-Strike: Global Offensive|65376795    |8658023     |7.551007314256384 |
|Garry's Mod                     |14641080    |509966      |28.709913994266284|
|Grand Theft Auto V              |13521915    |2347169     |5.760946484893077 |
|Terraria                        |13191243    |290940      |45.34008042895442 |
|Left 4 Dead 2                   |12876720    |336560      |38.25980508676016 |
|Team Fortress 2                 |11849698    |803922      |14.739860334709089|
|Dota 2                          |10744265    |2225412     |4.82798915436782  |
|Tom Clancy's Rainbow Six Siege  |10372010    |1575717     |6.582406612354884 |
|Rust                            |10255189    |1570162     |6.531293586266894 |
|Rocket League                   |992998

In absolute value, Counter-Strike: Global Offensive has more positive advices (65,4M). Yet Terraria has a better ratio : 45,34 positive advices for 1 negative advice against 7,5 onpour Counter-Strike. Then, Terraria is proportianaly more liked.

Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
from pyspark.sql.functions import (
    col, year, try_to_date, count, countDistinct
)

result = (flat_df \
    .select("name", "release_date") \
    .distinct() \
    .withColumn( \
        "release_year", \
        year(try_to_date(col("release_date"), "yyyy/M/d")) \
    ) \
    .filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .agg( \
        count("*").alias("release_nb"), \
        countDistinct("name").alias("name_nb") \
    ) \
    .filter((col('release_nb') > 0) & (col('name_nb') > 0)) \
    .withColumn('ratio', col('release_nb') /  col('name_nb')) \
    .orderBy(desc("release_year")) \
)

result.show()

+------------+----------+-------+------------------+
|release_year|release_nb|name_nb|             ratio|
+------------+----------+-------+------------------+
|        2022|      7401|   7394|1.0009467135515282|
|        2021|      8676|   8668|1.0009229349330873|
|        2020|      8156|   8147|1.0011047011169756|
|        2019|      6797|   6793|1.0005888414544384|
|        2018|      7508|   7500|1.0010666666666668|
|        2017|      5846|   5843|1.0005134348793427|
|        2016|      4089|   4088|1.0002446183953033|
|        2015|      2523|   2522| 1.000396510705789|
|        2014|      1524|   1524|               1.0|
|        2013|       462|    462|               1.0|
|        2012|       341|    341|               1.0|
|        2011|       266|    266|               1.0|
|        2010|       280|    280|               1.0|
|        2009|       309|    309|               1.0|
|        2008|       158|    158|               1.0|
|        2007|        97|     97|             

Yes. There are years with more releases. 
- In absolute value, 2014 to 2022. There are more releases after the covid's year in 2021 with 8676. 
- In prortional value, 2015 to 2022. Releases become higher than games since 2015. Yet, releases are higher on 2020, the covid's year with 1.0011 releases for one game.  
We observe that Covid accentuated the trend which returns then at normal pace in 2022 with 7401 and 1.00094.

In [0]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-------

In [0]:
clean_df.printSchema()

root
 |-- ccu: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- category: string (nullable = true)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)



How are the prizes distributed? Are there many games with a discount?

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'name', 'release_date', 'price', 'initialprice', 'discount') \
    .distinct() \
    .filter(clean_df.discount != 0 ) \
    .orderBy('publisher', 'name', 'release_date') \
    .show(truncate=False)

+-----------------------+-------------------------------------------------------+------------+-----+------------+--------+
|publisher              |name                                                   |release_date|price|initialprice|discount|
+-----------------------+-------------------------------------------------------+------------+-----+------------+--------+
|                       |breaking gates                                         |2020/11/17  |244  |699         |65      |
|                       |esports life tycoon                                    |2020/09/3   |1199 |1999        |40      |
|                       |goat of duty                                           |2019/07/10  |599  |999         |40      |
|                       |seven wonders of st. clementine                        |2020/10/30  |99   |999         |90      |
|                       |song of horror complete edition                        |2019/10/31  |1799 |2999        |40      |
|               

Prices don't change ? There is no discount on leaders publisher.

What are the most represented languages?

Are there many games prohibited for children under 16/18?